In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aadarshvani/100-unique-qa-dataset")

print("Path to dataset files:", path)

/Users/vaishnavishinde/anaconda3/envs/agdenv/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/Users/vaishnavishinde/anaconda3/envs/agdenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 2.04k/2.04k [00:00<00:00, 708kB/s]

Extracting model files...
Path to dataset files: /Users/vaishnavishinde/.cache/kagglehub/datasets/aadarshvani/100-unique-qa-dataset/versions/1


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv(path + "/100_Unique_QA_Dataset.csv")

In [5]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [7]:
# Tokenize

def tokenize(text):
    text = text.lower()
    text = text.replace("?", "")
    text = text.replace("'", "")
    return text.split()

In [16]:
# Vocabulary

vocab= {'<UNK>': 0}

In [17]:
# build vocab

def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])

    merged_tokens = tokenized_question + tokenized_answer

    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

In [18]:
df.apply(build_vocab, axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [20]:
len(vocab)

324

In [22]:
# convert words to numerical indices

def text_to_indices(text):
    indexed_text = []

    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    return indexed_text

In [26]:
text_to_indices("capital of india")

[4, 5, 73]

In [27]:
import torch
from torch.utils.data import Dataset, DataLoader

In [29]:
class QADataset(Dataset):

    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab
    
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, idx):
        question = self.df.iloc[idx]['question']
        answer = self.df.iloc[idx]['answer']

        question_indices = text_to_indices(question)
        answer_indices = text_to_indices(answer)

        return torch.tensor(question_indices), torch.tensor(answer_indices)

In [30]:
dataset = QADataset(df, vocab)

In [33]:
dataset[34]

(tensor([  1,   2,   3,  37, 133,   5,  26]), tensor([134]))

In [34]:
dataloader = DataLoader(dataset, batch_size = 1, shuffle = True)

In [48]:
for question, answer in dataloader:
    print(question, answer[0])

tensor([[ 1,  2,  3, 69,  5,  3, 70, 71]]) tensor([72])
tensor([[ 10,  29, 130, 131]]) tensor([132])
tensor([[10, 96,  3, 97]]) tensor([98])
tensor([[ 42, 101,   2,   3,  17]]) tensor([102])
tensor([[  1,   2,   3, 221,   5, 222, 223, 224]]) tensor([225])
tensor([[10, 55,  3, 56,  5, 57]]) tensor([58])
tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([74])
tensor([[  1,   2,   3,   4,   5, 135]]) tensor([136])
tensor([[  1,   2,   3,  37, 133,   5,  26]]) tensor([134])
tensor([[ 42, 117, 118,   3, 119,  94, 120]]) tensor([121])
tensor([[ 78,  79, 288,  81,  19,  14, 289]]) tensor([85])
tensor([[ 10,  75,   3, 296,  19, 297]]) tensor([298])
tensor([[ 10,  75, 111]]) tensor([112])
tensor([[ 42, 125,   2,  62,  63,   3, 126, 127]]) tensor([128])
tensor([[ 42, 255,   2, 256,  83, 257, 258]]) tensor([259])
tensor([[ 1,  2,  3,  4,  5, 99]]) tensor([100])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([52])
tensor([[42, 43, 44, 45, 46, 47, 48]]) tensor([49])
tensor([[ 42, 200,   2,  14, 201, 202

In [37]:
import torch.nn as nn

In [58]:
class QAModel(nn.Module):
    
    def __init__(self, vocab_size):
        super(QAModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim = 50)
        self.rnn = nn.RNN(input_size = 50, hidden_size = 64, batch_first = True)
        self.fc = nn.Linear(64, vocab_size)
        pass

    def forward(self, question):
        embedded_question = self.embedding(question)
        hidden, final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))
        return output

In [59]:
learning_rate = 0.001
epochs = 20

In [60]:
model = QAModel(len(vocab))

In [61]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [64]:
for epoch in range(epochs):
    total_loss = 0
    for question, answer in dataloader:

        optimizer.zero_grad()

        # forward pass
        output = model(question)

        # compute loss
        loss = criterion(output, answer[0])

        # backward pass
        loss.backward()

        # update parameters
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader)}")

Epoch 1/20, Loss: 0.15552665326330398
Epoch 2/20, Loss: 0.13625342051188152
Epoch 3/20, Loss: 0.11904157594674164
Epoch 4/20, Loss: 0.10383476246562269
Epoch 5/20, Loss: 0.09248851024442249
Epoch 6/20, Loss: 0.08259985223412514
Epoch 7/20, Loss: 0.07336528897285462
Epoch 8/20, Loss: 0.06568688133524524
Epoch 9/20, Loss: 0.059122363208896585
Epoch 10/20, Loss: 0.053066680228544605
Epoch 11/20, Loss: 0.04849507423738639
Epoch 12/20, Loss: 0.04434700361970398
Epoch 13/20, Loss: 0.040721527735392254
Epoch 14/20, Loss: 0.037312375153932306
Epoch 15/20, Loss: 0.03436954265667332
Epoch 16/20, Loss: 0.03171261331687371
Epoch 17/20, Loss: 0.029341683712684447
Epoch 18/20, Loss: 0.027098710079573922
Epoch 19/20, Loss: 0.02525667180824611
Epoch 20/20, Loss: 0.023475378700014617


In [88]:
def predict(model, question, threshold = 0.5):

    # convert question to indices
    question_indices = text_to_indices(question)
    # convert indices to tensor
    question_tensor = torch.tensor(question_indices).unsqueeze(0)

    # send to model
    output = model(question_tensor)

    # convert logit to probabilities
    prob = torch.nn.functional.softmax(output, dim = 1)

    # find index of max probability
    value, index = torch.max(prob, dim = 1)

    if value < threshold:
        print("Sorry, I don't know the answer to that question.")
    else:
        print("Answer:", list(vocab.keys())[index])

In [90]:
predict(model, "capital of india")

Answer: delhi


In [91]:
predict(model, 'what is your name?')

Sorry, I don't know the answer to that question.
